# NBA Model v2 — easy Colab runner

This notebook has only **two cells to run**:

1. Run **Create a forecast** once. It mounts Drive, installs the project, refreshes or reuses your data, trains Model v2, and creates the next usable forecast.
2. Run **Ask a player question** as many times as you want.

Every project log and error is shown live while a cell is running.

> Tip: choose **Runtime → Run all**. The defaults are the safest easy path: a quick data refresh, a degraded non-published forecast, and all reusable files saved in Google Drive. Quick refresh never downloads thousands of player bios. Choose **current_files** to train from the data files already in Drive without refreshing them.

Leave `RUN_DATE` blank and the notebook automatically selects the next game date that satisfies the forecast cutoff. `reuse_latest` uses the last frozen snapshot, while `current_files` freezes the current CSV files first. The notebook always prints the snapshot it selected.

The lightweight baseline intentionally cannot be promoted. Official and shadow modes keep the stricter point-in-time roster rules; use them only when you understand those requirements.


In [ ]:
# @title 1) Create a forecast { display-mode: "form" }
# @markdown The defaults work without any model or command-line knowledge.

# --- The only settings most people need ------------------------------------
MODE = "degraded"  # @param ["degraded", "shadow", "official"]
HORIZON = "morning"  # @param ["previous_night", "morning", "pregame_90m", "pregame_30m"]
RUN_DATE = ""  # @param {type:"date"}
SIMULATIONS = 100  # @param {type:"integer"}
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}
DATA_REFRESH = "quick"  # @param ["quick", "current_files", "reuse_latest", "full"]
# @markdown Leave `RUN_DATE` blank to use the next forecastable NBA game date.
# @markdown `current_files` trains from the data already in Drive. `reuse_latest` uses the last frozen copy.

# --- Advanced settings (the defaults normally should not be changed) -------
REUSE_SNAPSHOT_ID = ""
DRIVE_FOLDER = "nba_model"
REPO_URL = "https://github.com/jxylxnn/knowing.git"
REPO_REF = "V2-WIP"
SEED = 42

import json
import os
import subprocess
import sys
import threading
from datetime import date, datetime
from pathlib import Path
from zoneinfo import ZoneInfo

for stale_name in ("FORECAST_PATH", "SAMPLES_PATH", "CANDIDATE_DIR", "TARGET_DATE"):
    globals().pop(stale_name, None)


def run(command, *, cwd=None):
    """Run one command while streaming and retaining every output line."""
    print("Working…", flush=True)
    process = subprocess.Popen(
        [str(item) for item in command],
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        bufsize=1,
    )
    stdout_lines, stderr_lines = [], []

    def pump(pipe, bucket):
        for line in iter(pipe.readline, ""):
            bucket.append(line)
            print(line, end="", flush=True)
        pipe.close()

    threads = [
        threading.Thread(target=pump, args=(process.stdout, stdout_lines)),
        threading.Thread(target=pump, args=(process.stderr, stderr_lines)),
    ]
    for thread in threads:
        thread.start()
    returncode = process.wait()
    for thread in threads:
        thread.join()
    result = subprocess.CompletedProcess(
        command, returncode, "".join(stdout_lines), "".join(stderr_lines)
    )
    if returncode:
        raise RuntimeError(
            f"A step failed (exit code {returncode}). "
            "Read the message immediately above for the exact cause."
        )
    return result


def last_nonempty_line(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else ""


def season_for(day):
    start_year = day.year if day.month >= 7 else day.year - 1
    return f"{start_year}-{str(start_year + 1)[-2:]}"


if MODE not in {"degraded", "shadow", "official"}:
    raise ValueError(f"Unsupported MODE: {MODE}")
if HORIZON not in {"previous_night", "morning", "pregame_90m", "pregame_30m"}:
    raise ValueError(f"Unsupported HORIZON: {HORIZON}")
if int(SIMULATIONS) < 1:
    raise ValueError("SIMULATIONS must be at least 1.")
if DATA_REFRESH not in {"quick", "current_files", "reuse_latest", "full"}:
    raise ValueError(f"Unsupported DATA_REFRESH: {DATA_REFRESH}")

print("STEP 1/7 — Preparing Colab")
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive  # type: ignore
    except ImportError as exc:
        raise RuntimeError("Google Drive mounting is available only in Colab.") from exc
    drive.mount("/content/drive")
    STORAGE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER.strip()
else:
    STORAGE_ROOT = Path("/content/nba_model")
    print("Drive is off; files will disappear when this Colab session ends.")

REPO_PATH = Path("/content/knowing").resolve()
DATA_DIR = STORAGE_ROOT / "data"
MODELS_DIR = STORAGE_ROOT / "models"
OUTPUT_DIR = STORAGE_ROOT / "sim_results" / "v2"
for folder in (DATA_DIR, MODELS_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("\nSTEP 2/7 — Getting Model v2")
if not (REPO_PATH / ".git").is_dir():
    run(["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, REPO_PATH])
else:
    run(["git", "fetch", "origin", REPO_REF], cwd=REPO_PATH)
    run(["git", "checkout", REPO_REF], cwd=REPO_PATH)
    run(["git", "merge", "--ff-only", f"origin/{REPO_REF}"], cwd=REPO_PATH)

required_files = (
    "train.py",
    "update_data.py",
    "canonicalize_data.py",
    "check_data.py",
    "capture_official.py",
    "rebuild_snapshot.py",
    "simulate_season.py",
    "query_prob.py",
    "config/model_v2.yaml",
)
missing = [name for name in required_files if not (REPO_PATH / name).is_file()]
if missing:
    raise RuntimeError(
        f"The selected repository ref does not contain Model v2: {', '.join(missing)}"
    )

packages = [
    "numpy==2.2.6", "pandas==2.2.3", "pyarrow==24.0.0",
    "scipy==1.16.3", "scikit-learn==1.8.0", "PyYAML==6.0.3",
    "joblib==1.5.3", "psutil==7.2.2", "nba_api==1.11.3",
    "requests==2.32.4", "curl-cffi==0.16.3",
]
run([sys.executable, "-m", "pip", "install", "--quiet", *packages], cwd=REPO_PATH)
os.chdir(REPO_PATH)
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

print("\nSTEP 3/7 — Preparing a frozen data snapshot")
manifest_dir = DATA_DIR / "manifests"
if REUSE_SNAPSHOT_ID.strip():
    SNAPSHOT_ID = REUSE_SNAPSHOT_ID.strip()
    manifest_path = DATA_DIR / "manifests" / f"source_snapshot_{SNAPSHOT_ID}.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Snapshot not found: {SNAPSHOT_ID}")
    print("Reusing snapshot:", SNAPSHOT_ID)
elif DATA_REFRESH == "current_files":
    from src.data.snapshots import create_source_snapshot

    missing_current = [
        name for name in ("nba_players.csv", "nba_games.csv")
        if not (DATA_DIR / name).is_file()
    ]
    if missing_current:
        raise RuntimeError(
            "Cannot train from current files; missing: "
            + ", ".join(missing_current)
            + ". Choose DATA_REFRESH='quick' to download data first."
        )
    current_manifest = create_source_snapshot(DATA_DIR)
    SNAPSHOT_ID = current_manifest.snapshot_id
    print("Frozen current Drive data (no game-log API calls):", SNAPSHOT_ID)
elif DATA_REFRESH == "reuse_latest":
    manifests = list(manifest_dir.glob("source_snapshot_*.json"))
    if not manifests:
        raise RuntimeError(
            "No saved snapshot exists yet. Choose DATA_REFRESH='quick' for the first run."
        )
    latest_manifest = max(manifests, key=lambda path: path.stat().st_mtime)
    SNAPSHOT_ID = latest_manifest.stem.removeprefix("source_snapshot_")
    print("Reusing latest snapshot (no game-log API calls):", SNAPSHOT_ID)
else:
    update_command = [
        sys.executable, "update_data.py", "--update", "--snapshot",
        "--data-dir", DATA_DIR,
    ]
    if DATA_REFRESH == "quick":
        update_command.extend(
            ["--bio-mode", "cached", "--request-timeout", "10"]
        )
        print("Quick refresh: at most two 10-second game-log waits; saved bios only.")
    else:
        print("Full refresh selected: missing player bios may take a long time.")
    run(
        update_command,
        cwd=REPO_PATH,
    )
    manifests = list(manifest_dir.glob("source_snapshot_*.json"))
    if not manifests:
        raise RuntimeError("The data update did not create a source snapshot.")
    latest_manifest = max(manifests, key=lambda path: path.stat().st_mtime)
    SNAPSHOT_ID = latest_manifest.stem.removeprefix("source_snapshot_")
    print("Created snapshot:", SNAPSHOT_ID)

print("\nSTEP 4/7 — Capturing the official schedule")
eastern_today = datetime.now(ZoneInfo("America/New_York")).date()
requested_day = date.fromisoformat(RUN_DATE) if RUN_DATE.strip() else eastern_today
CAPTURE_SEASON = season_for(requested_day)
captures = []
schedule_capture = run(
    [
        sys.executable, "capture_official.py", "--native", "schedule",
        "--season", CAPTURE_SEASON, "--data-dir", DATA_DIR,
    ],
    cwd=REPO_PATH,
)
captures.append(last_nonempty_line(schedule_capture.stdout))

if MODE in {"shadow", "official"}:
    print("Strict mode selected: capturing all 30 current team rosters.")
    roster_capture = run(
        [
            sys.executable, "capture_official.py", "--native", "rosters",
            "--season", CAPTURE_SEASON, "--data-dir", DATA_DIR,
        ],
        cwd=REPO_PATH,
    )
    captures.append(last_nonempty_line(roster_capture.stdout))

rebuild = [
    sys.executable, "rebuild_snapshot.py", "--base-snapshot-id", SNAPSHOT_ID,
    "--data-dir", DATA_DIR,
]
for capture in captures:
    rebuild.extend(["--capture", capture])
rebuilt = run(rebuild, cwd=REPO_PATH)
SNAPSHOT_ID = last_nonempty_line(rebuilt.stdout)
if not SNAPSHOT_ID:
    raise RuntimeError("Snapshot rebuild did not return a snapshot id.")
print("Forecast snapshot:", SNAPSHOT_ID)

print("\nSTEP 5/7 — Choosing a game date and checking the snapshot")
from src.data.snapshots import load_source_snapshot_manifest
from src.features.snapshot_inputs import load_snapshot_schedule
from src.simulation.v2_runner import horizon_cutoff
import pandas as pd

manifest = load_source_snapshot_manifest(DATA_DIR, SNAPSHOT_ID)
available_times = [manifest.created_datetime]
available_times.extend(
    pd.Timestamp(record.fetched_at or manifest.created_at).to_pydatetime()
    for record in manifest.files
)
snapshot_ready_at = max(available_times)
schedule = load_snapshot_schedule(DATA_DIR, SNAPSHOT_ID).copy()
schedule["_game_date"] = pd.to_datetime(schedule["GAME_DATE"]).dt.date.astype(str)

if RUN_DATE.strip():
    TARGET_DATE = RUN_DATE.strip()
else:
    TARGET_DATE = None
    today_text = eastern_today.isoformat()
    for game_date in sorted(schedule["_game_date"].unique()):
        if game_date < today_text:
            continue
        slate = schedule.loc[schedule["_game_date"] == game_date]
        cutoffs = [
            horizon_cutoff(pd.Timestamp(tip).to_pydatetime(), HORIZON)
            for tip in slate["SCHEDULED_TIP"]
        ]
        if cutoffs and all(snapshot_ready_at <= cutoff for cutoff in cutoffs):
            TARGET_DATE = game_date
            break
    if TARGET_DATE is None:
        raise RuntimeError(
            f"No upcoming game date in the {CAPTURE_SEASON} schedule can use a "
            f"fresh {HORIZON} snapshot. Enter a future RUN_DATE or try another horizon."
        )

selected_slate = schedule.loc[schedule["_game_date"] == TARGET_DATE]
if selected_slate.empty:
    raise RuntimeError(
        f"The captured {CAPTURE_SEASON} schedule has no games on {TARGET_DATE}. "
        "Clear RUN_DATE to choose the next available slate automatically."
    )
selected_cutoffs = [
    horizon_cutoff(pd.Timestamp(tip).to_pydatetime(), HORIZON)
    for tip in selected_slate["SCHEDULED_TIP"]
]
if any(snapshot_ready_at > cutoff for cutoff in selected_cutoffs):
    raise RuntimeError(
        f"The fresh snapshot is too late for at least one {HORIZON} cutoff on "
        f"{TARGET_DATE}. Clear RUN_DATE to select the next valid slate automatically."
    )

if MODE in {"shadow", "official"}:
    strict_cutoff_dates = {cutoff.date() for cutoff in selected_cutoffs}
    if strict_cutoff_dates != {eastern_today}:
        expected = ", ".join(sorted(day.isoformat() for day in strict_cutoff_dates))
        raise RuntimeError(
            f"{MODE.title()} mode requires roster capture on the forecast cutoff's "
            f"Eastern date ({expected}). Today is {eastern_today}. Run this notebook "
            "on the required date, or use degraded mode now."
        )

run(
    [
        sys.executable, "canonicalize_data.py", "--snapshot-id", SNAPSHOT_ID,
        "--data-dir", DATA_DIR,
    ],
    cwd=REPO_PATH,
)
run(
    [
        sys.executable, "check_data.py", "--snapshot-id", SNAPSHOT_ID,
        "--data-dir", DATA_DIR,
    ],
    cwd=REPO_PATH,
)

print("\nSTEP 6/7 — Training the immutable Model v2 baseline")
trained = run(
    [
        sys.executable, "train.py", "--architecture", "v2",
        "--preset", "baseline", "--snapshot-id", SNAPSHOT_ID,
        "--data-dir", DATA_DIR, "--models-dir", MODELS_DIR,
        "--config", "config/model_v2.yaml", "--json",
    ],
    cwd=REPO_PATH,
)
try:
    training_result = json.loads(trained.stdout)
    CANDIDATE_DIR = str(Path(training_result["candidate"]).resolve())
except (json.JSONDecodeError, KeyError) as exc:
    raise RuntimeError("Training finished without reporting a candidate bundle.") from exc

print("\nSTEP 7/7 — Creating the forecast")
simulate = [
    sys.executable, "simulate_season.py", "--date", TARGET_DATE,
    "--snapshot-id", SNAPSHOT_ID, "--horizon", HORIZON,
    "--data-dir", DATA_DIR, "--models-dir", MODELS_DIR,
    "--output-dir", OUTPUT_DIR, "--sims", str(int(SIMULATIONS)),
    "--seed", str(int(SEED)), "--json",
]
if MODE in {"degraded", "shadow"}:
    simulate.extend(["--candidate", CANDIDATE_DIR])
if MODE == "degraded":
    simulate.append("--allow-degraded")
simulated = run(simulate, cwd=REPO_PATH)
try:
    forecast_result = json.loads(simulated.stdout)
    new_forecast_path = forecast_result["forecast_path"]
    new_samples_path = forecast_result["samples_path"]
except (json.JSONDecodeError, KeyError) as exc:
    raise RuntimeError("Simulation finished without reporting forecast files.") from exc
if not Path(new_forecast_path).is_file() or not Path(new_samples_path).is_file():
    raise RuntimeError("Simulation reported forecast files that do not exist.")
FORECAST_PATH = new_forecast_path
SAMPLES_PATH = new_samples_path

print("\n" + "═" * 68)
print("FORECAST READY")
print("Date       :", TARGET_DATE)
print("Mode       :", forecast_result.get("mode"))
print("Games      :", forecast_result.get("games"))
print("Forecasts  :", FORECAST_PATH)
print("Samples    :", SAMPLES_PATH)
print("Next       : edit and run the player-question cell below.")
print("═" * 68)


In [ ]:
# @title 2) Ask a player question { display-mode: "form" }
# @markdown Enter a player, stat, and line. Leave `PLAYER` blank for an automatic example.
PLAYER = ""  # @param {type:"string"}
STAT = "pts"  # @param ["pts", "reb", "ast", "stl", "blk", "tov"]
LINE = 25.5  # @param {type:"number"}

import json
import sys
from pathlib import Path
import pandas as pd

if "FORECAST_PATH" not in globals() or not Path(FORECAST_PATH).is_file():
    raise RuntimeError("Run the Create a forecast cell first.")

selected_player = PLAYER.strip()
if not selected_player:
    preview = pd.read_parquet(FORECAST_PATH, columns=["PLAYER_ID", "STAT", "MEAN"])
    preview = preview.loc[preview["STAT"].astype(str).str.casefold() == STAT.casefold()]
    if preview.empty:
        raise RuntimeError(f"The forecast has no {STAT} rows to query.")
    selected_player = str(preview.sort_values("MEAN", ascending=False).iloc[0]["PLAYER_ID"])

result = run(
    [
        sys.executable, "query_prob.py", "--player", selected_player,
        "--stat", STAT, "--line", str(LINE),
        "--forecast-file", FORECAST_PATH,
        "--players-file", str(DATA_DIR / "nba_players.csv"),
        "--json",
    ],
    cwd=REPO_PATH,
)

answer = json.loads(result.stdout)
player_label = PLAYER.strip() or f"Player {answer['player_id']}"
names_path = DATA_DIR / "nba_players.csv"
if not PLAYER.strip() and names_path.is_file():
    names = pd.read_csv(names_path, usecols=["PLAYER_ID", "PLAYER_NAME"])
    match = names.loc[names["PLAYER_ID"].astype(str) == str(answer["player_id"])]
    if not match.empty:
        player_label = str(match.iloc[-1]["PLAYER_NAME"])
quality = "exact" if answer.get("probability_exact") else "estimated"
print(f"\n{player_label} — {STAT.upper()} line {float(LINE):g}")
print(f"Expected value : {answer['mean']:.2f}")
print(f"Over           : {answer['over']:.1%}")
print(f"Under          : {answer['under']:.1%}")
print(f"Push           : {answer['push']:.1%}")
print(f"Probability    : {quality}")
